# 1. ChatPromptTemplate 的高级特性

## 1.1 部分变量预填充：partial()

预填充某些固定不变的变量，创建模板的变体。
使用场景：
* 某些变量在所有调用中都相同
* 需要为不同用户/场景创建定制模板

举例1:


In [1]:
from langchain_core.prompts import ChatPromptTemplate

# 原始模版
template = ChatPromptTemplate.from_messages([
    ("system", "你是{role}，目标用户是{audience}"),
    ("user", "{task}")
])

# 部分填充
customer_support_template = template.partial(
    role="客服专员",
    audience="普通用户"
)
# 现在只需要提供 task
messages = customer_support_template.invoke({"task":"解释退款政策"})
print(messages)


messages=[SystemMessage(content='你是客服专员，目标用户是普通用户', additional_kwargs={}, response_metadata={}), HumanMessage(content='解释退款政策', additional_kwargs={}, response_metadata={})]


In [2]:
# 场景：为不同部门创建专用模板
base_template = ChatPromptTemplate.from_messages([
    ("system", "你是{department}的{role}"),
    ("user", "{task}")
])
# IT 部门
it_template = base_template.partial(
    department="IT 部门",
    role="技术支持"
)
# 销售部门
sales_template = base_template.partial(
    department="销售部门",
    role="销售顾问"
)
sales_template.invoke({"task":"为什么每年年底汽车会促销"})

ChatPromptValue(messages=[SystemMessage(content='你是销售部门的销售顾问', additional_kwargs={}, response_metadata={}), HumanMessage(content='为什么每年年底汽车会促销', additional_kwargs={}, response_metadata={})])

## 1.2 消息占位符

当你不确定消息提示模板使用什么角色，或者希望在格式化过程中 插入消息列表 时，该怎么办？ 这就
需要使用消息占位符，负责在特定位置添加消息列表。

使用场景：多轮对话系统存储历史消息以及Agent的中间步骤处理此功能非常有用。

方式1：JSON形式

In [4]:
from langchain_core.prompts import ChatPromptTemplate
template = ChatPromptTemplate.from_messages([
    ("system", "你是一个有用的AI助手"),
    ("placeholder", "{conversation}"),
])

prompt_value = template.invoke(
    {
        "conversation": [
            ("human", "你好"),
            ("assistant", "你好！有什么我可以帮助你的吗？"),
            ("human", "我想知道人工智能英文怎么说"),
            ("assistant", "人工智能英文说：Artificial Intelligence")
        ]
    }
)

print(prompt_value)

messages=[SystemMessage(content='你是一个有用的AI助手', additional_kwargs={}, response_metadata={}), HumanMessage(content='你好', additional_kwargs={}, response_metadata={}), AIMessage(content='你好！有什么我可以帮助你的吗？', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='我想知道人工智能英文怎么说', additional_kwargs={}, response_metadata={}), AIMessage(content='人工智能英文说：Artificial Intelligence', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]


方式2：MessagesPlaceholder实例

In [7]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage
prompt_template = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant"),
    MessagesPlaceholder("msgs")
])
prompt_template.invoke({"msgs": [HumanMessage(content="hi! LiHua"), AIMessage(content="hello! what can I help you with?")]})
# prompt_template.format_messages(msgs=[HumanMessage(content="hi!")])

ChatPromptValue(messages=[SystemMessage(content='You are a helpful assistant', additional_kwargs={}, response_metadata={}), HumanMessage(content='hi! LiHua', additional_kwargs={}, response_metadata={}), AIMessage(content='hello! what can I help you with?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])])

这将生成两条消息，第一条是系统消息，第二条是我们传入的 HumanMessage。 如果我们传入了 5 条
消息，那么总共会生成 6 条消息（系统消息加上传入的 5 条消息）。 这对于将一系列消息插入到特定位置非常有用。

举例2：存储对话历史内容

In [8]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
prompt_template = ChatPromptTemplate.from_messages([
    ("system", "你是一个非常友好的AI助手"),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{question}")
])

prompt_template.invoke(
    {
        "history": [
            ("human", "6 + 2 = ?"),
            ("ai", "6 + 2 = 8")
        ],
        "question": "结果再加10呢?"
    }
)


ChatPromptValue(messages=[SystemMessage(content='你是一个非常友好的AI助手', additional_kwargs={}, response_metadata={}), HumanMessage(content='6 + 2 = ?', additional_kwargs={}, response_metadata={}), AIMessage(content='6 + 2 = 8', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='结果再加10呢?', additional_kwargs={}, response_metadata={})])

## 1.3 可复用模板库

在实际项目中，建议创建模板库。

举例1：

templates.py文件声明如下
